# ขั้นตอนที่ 1: Import Dependencies

In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

# ขั้นตอนที่ 2: ตั้งค่า MediaPipe Holistic และฟังก์ชั่นตรวจจับ

In [2]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR CONVERSION RGB 2 BGR
    return image, results

def draw_styled_landmarks(image, results):
    # Draw face connections
    if results.face_landmarks:
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
    # Draw pose connections
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )
    # Draw left hand connections
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )
    # Draw right hand connections
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )

# ขั้นตอนที่ 3: ดึงพิกัด Keypoints (Extract Keypoint Values)

In [3]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

# ขั้นตอนที่ 4: ตั้งค่าโฟลเดอร์สำหรับเก็บข้อมูล (🛠️ จุดแก้ที่ 1)

In [6]:
# Path for exported data, in numpy arrays
DATA_PATH = os.path.join('MP_DATA') 

# Actions that we try to detect (25 คำเดิม + 1 idle)
actions = np.array([
    'idle', 'Hello', 'How', 'You', 'Your', 'Name', 'What', 'My', "I'm", 
    'Fine', 'Meet you', 'Nice', 'Thanks', 'I Love You', 
    'COOL!', 'Again', 'Yes', 'No', 'Good', 'Bad', 'Mornig',
])

# เพิ่มจำนวนวิดีโออัดเก็บเป็น 40 คลิป ต่อ 1 คำ เพื่อเพิ่มความแม่นยำ
no_sequences = 40
sequence_length = 30

# Create directories
for action in actions: 
    for sequence in range(no_sequences):
        try: 
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

# ขั้นตอนที่ 5: บันทึกข้อมูลพิกัดมือ/ร่างกายจาก Webcam

In [5]:
cap = cv2.VideoCapture(0)

# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    # Loop through actions
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                # Read feed
                ret, frame = cap.read()

                # Make detections
                image, results = mediapipe_detection(frame, holistic)

                # Draw landmarks
                draw_styled_landmarks(image, results)
                
                # Apply wait logic
                if frame_num == 0: 
                    cv2.putText(image, 'STARTING COLLECTION', (120,200), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, f'Collecting frames for {action} Video Number {sequence}', (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(300)
                else: 
                    cv2.putText(image, f'Collecting frames for {action} Video Number {sequence}', (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                
                # Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break
                    
    cap.release()
    cv2.destroyAllWindows()

# ขั้นตอนที่ 6: จัดเตรียม Data สำหรับ Train โมเดล

In [6]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

label_map = {label:num for num, label in enumerate(actions)}

sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), f"{frame_num}.npy"))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

X = np.array(sequences)
y = to_categorical(labels).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

# ขั้นตอนที่ 7: สร้างและ Train โมเดล LSTM (🛠️ จุดแก้ที่ 2)

In [7]:
import os
import numpy as np
import tensorflow as tf

# ใช้ tf.keras ป้องกัน ModuleNotFoundError
Sequential = tf.keras.models.Sequential
LSTM = tf.keras.layers.LSTM
Dense = tf.keras.layers.Dense
TensorBoard = tf.keras.callbacks.TensorBoard

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
# ปรับขยายขนาด Dense Layer รองรับ 26 คลาส
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

model.fit(X_train, y_train, epochs=200, callbacks=[tb_callback])

# Save weights
model.save('action.h5')

c:\Users\epryw\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - categorical_accuracy: 0.0504 - loss: 3.1882
Epoch 2/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - categorical_accuracy: 0.0647 - loss: 3.1445
Epoch 3/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - categorical_accuracy: 0.0691 - loss: 2.8692
Epoch 4/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - categorical_accuracy: 0.0779 - loss: 2.7350
Epoch 5/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - categorical_accuracy: 0.1086 - loss: 2.6695
Epoch 6/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - categorical_accuracy: 0.1338 - loss: 2.5707
Epoch 7/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - categorical_accuracy: 0.1173 - loss: 2.5934
Epoch 8/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - categorical_accuracy: 0.1404 - loss: 2.5527
Epoch 9/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - categorical_accuracy: 0.1064 - loss: 2.6705
Epoch 10/200
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - categorical_accuracy: 0.1151 - loss: 2.4817
Epoch 11/200
29/29 

# ขั้นตอนที่ 8: ฟังก์ชั่นแสดงแถบความน่าจะเป็น (🛠️ จุดแก้ที่ 3)

In [7]:
# สุ่มสี BGR แบบสุ่มแต่คงที่ (26 สีสำหรับ 26 คำ)
np.random.seed(42)
colors = [tuple(np.random.randint(0, 256, size=3).tolist()) for _ in range(len(actions))]

def prob_vis(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    # ปรับขนาดและระยะห่างบรรทัดให้พอดีจอ
    for num, prob in enumerate(res):
        color = colors[num]
        cv2.rectangle(output_frame, (0, 45 + num * 22), (int(prob * 100), 65 + num * 22), color, -1)
        cv2.putText(output_frame, actions[num], (0, 60 + num * 22), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    return output_frame

# ขั้นตอนที่ 9: ทดสอบรัน Real-time แบบเสถียร (🛠️ จุดแก้ที่ 4 & 5)

In [ ]:
# # 1. New detection variables
# sequence = []
# sentence = []
# predictions = []
# threshold = 0.9  # ปรับเพิ่มเป็น 0.9 ให้ระบบแสดงผลเฉพาะตอนที่มั่นใจจริงๆ

# cap = cv2.VideoCapture(0)

# # Set mediapipe model 
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
#     while cap.isOpened():

#         # Read feed
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Make detections
#         image, results = mediapipe_detection(frame, holistic)
        
#         # Draw landmarks
#         draw_styled_landmarks(image, results)
        
#         # 2. Prediction logic
#         keypoints = extract_keypoints(results)
#         sequence.append(keypoints)
#         sequence = sequence[-30:]
        
#         if len(sequence) == 30:
#             res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
#             predictions.append(np.argmax(res))
            
#             # 3. Viz logic
#             if np.unique(predictions[-10:])[0] == np.argmax(res):
#                 if res[np.argmax(res)] > threshold:
#                     predicted_action = actions[np.argmax(res)]
                    
#                     # กรองไม่ให้เอาคำว่า 'idle' เข้าประโยค
#                     if predicted_action != 'idle':
#                         if len(sentence) > 0:
#                             if predicted_action != sentence[-1]:
#                                 sentence.append(predicted_action)
#                         else:
#                             sentence.append(predicted_action)

#             if len(sentence) > 5:
#                 sentence = sentence[-5:]

#             # Visualize probabilities
#             image = prob_vis(res, actions, image, colors)
            
#         # แสดงประโยคด้านบน
#         cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
#         cv2.putText(image, ' '.join(sentence), (3,30), 
#                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
#         # Show to screen
#         cv2.imshow('OpenCV Feed', image)

#         # Break gracefully
#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break
            
#     cap.release()
#     cv2.destroyAllWindows()

# 1. New detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.65  # 🟢 ลดลงมาจาก 0.9 เพื่อให้จับคำได้ไวขึ้น

cap = cv2.VideoCapture(0)

# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()
        if not ret:
            break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        # 2. Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]
        
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            predictions.append(np.argmax(res))
            
            # 3. Viz logic
            # 🟢 ปรับเงื่อนไขให้แสดงผลทันทีเมื่อความมั่นใจเกิน threshold
            if res[np.argmax(res)] > threshold:
                predicted_action = actions[np.argmax(res)]
                
                # กรองไม่ให้เอาคำว่า 'idle' เข้าประโยค
                if predicted_action != 'idle':
                    if len(sentence) > 0:
                        if predicted_action != sentence[-1]:
                            sentence.append(predicted_action)
                    else:
                        sentence.append(predicted_action)

            if len(sentence) > 5:
                sentence = sentence[-5:]

            # Visualize probabilities
            image = prob_vis(res, actions, image, colors)
            
        # แสดงประโยคด้านบน
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

NameError: name 'model' is not defined

: 